# Forecast Probability Maps — CD and CW

Loads the ensemble compound-event forecast cubes produced by
`03_ecmwf_ensemble_processing.ipynb` and plots them alongside the observed
CD/CW classification for each high-impact event.

**Source notebook:** `probs_plots.ipynb`  
Cells 3 and 4 contained near-identical plotting loops (one with a hardcoded
obs path, one with a per-event obs path dictionary) — merged into one
parameterised function. Earlier draft layouts (cells 5–7) are not included.

**Inputs**
| File | Description |
|---|---|
| `output/forecasts/masked_HadUK_CD_final_{run_date_time}.nc` | CD ensemble forecast cube |
| `output/forecasts/masked_HadUK_CW_final_{run_date_time}.nc` | CW ensemble forecast cube |
| `output/obs/obs_data_{YYYYMMDD}_clean.pkl` | Observed CD/CW classification for the event day (from `02_sequence_footprints_plotting.ipynb`) |
| `data/scotland_rail.shp` | Rail network shapefile (read via shapefile library for lightweight plotting) |

**Outputs**  
One figure per event:  
`output/figures/forecast_prob_{event}.png`

**Configuration:** Edit the paths and event dictionary below.


In [ ]:
import os
import pickle
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import iris
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import shapefile
from shapely.geometry import shape

# ── User configuration ──────────────────────────────────────────────────────
FORECAST_DIR   = "output/forecasts"
OBS_DIR        = "output/obs"
RAIL_SHAPEFILE = "data/scotland_rail.shp"
FIG_DIR        = "output/figures"
os.makedirs(FIG_DIR, exist_ok=True)

# Map extent
MAP_EXTENT = [-8.0, -0.8, 54.5, 60.9]

# Forecast probability colourmap (purple-blue scale)
PROB_COLORS = ['white', 'lightgrey', '#E2B3CC', '#C49EE2', '#945794', '#44519F', '#423A69']
PROB_BINS   = [0, 1, 10, 20, 40, 60, 80, 100]

# Observation colourmap (binary: compound event or not)
OBS_BOUNDS = [0, 3, 4]   # values below 3 = "neither", 3 = compound

# Timestep indices within the forecast cube for each lead time
TIMESTEP_INDEX = {"3_days": 4, "5_days": 12, "10_days": 32}

# Events: keys are used for figure filenames and obs file lookup
# obs_date: YYYYMMDD string matching the obs pickle filename
EVENTS_DICT = {
    "Event_1_11Feb2021": {
        "obs_date": "20210211",
        "runs": [
            {"lead_time": "10_days", "run_date_time": "2021020100"},
            {"lead_time": "5_days",  "run_date_time": "2021020600"},
            {"lead_time": "3_days",  "run_date_time": "2021020800"},
        ],
    },
    "Event_2_28Feb2018": {
        "obs_date": "20180228",
        "runs": [
            {"lead_time": "10_days", "run_date_time": "2018021800"},
            {"lead_time": "5_days",  "run_date_time": "2018022300"},
            {"lead_time": "3_days",  "run_date_time": "2018022500"},
        ],
    },
    "Event_3_29Nov2010": {
        "obs_date": "20101129",
        "runs": [
            {"lead_time": "10_days", "run_date_time": "2010111900"},
            {"lead_time": "5_days",  "run_date_time": "2010112400"},
            {"lead_time": "3_days",  "run_date_time": "2010112600"},
        ],
    },
}

print("Configuration complete.")


## 1. Load shared resources

In [ ]:
# ── Colormaps ────────────────────────────────────────────────────────────────
cmap_prob = mpl.colors.ListedColormap(PROB_COLORS)
norm_prob  = mpl.colors.BoundaryNorm(boundaries=PROB_BINS, ncolors=len(cmap_prob.colors))

cmap_cd_obs = mpl.colors.ListedColormap(['white', 'magenta'])
cmap_cw_obs = mpl.colors.ListedColormap(['white', 'seagreen'])
norm_obs    = mpl.colors.BoundaryNorm(OBS_BOUNDS, cmap_cd_obs.N)

# ── Rail geometry (lightweight; avoids geopandas dependency in this notebook) ─
sf = shapefile.Reader(RAIL_SHAPEFILE)
rail_geometries = []
for shape_rec in sf.shapeRecords():
    geom = shape(shape_rec.shape)
    if geom.geom_type == 'LineString':
        rail_geometries.append(geom.coords)

def plot_rail(ax):
    """Overlay rail network lines on a Cartopy axes."""
    for line in rail_geometries:
        xs, ys = zip(*line)
        ax.plot(xs, ys, color='black', linewidth=0.75, alpha=0.9)

def set_map_extent(ax):
    ax.set_extent(MAP_EXTENT, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5)

print("Shared resources loaded.")


## 2. Core plotting function

In [ ]:
def plot_forecast_event(event_name, obs_date, run_entries):
    """
    Produce a combined forecast + observation figure for one event.

    Layout: 3 forecast columns (10-day, 5-day, 3-day) × 2 rows (CD, CW),
    plus one observation column on the right.

    Parameters
    ----------
    event_name  : str   Used for the figure title and filename
    obs_date    : str   YYYYMMDD string for locating the obs pickle
    run_entries : list  Dicts with 'lead_time' and 'run_date_time' keys
    """
    obs_path = os.path.join(OBS_DIR, f"obs_data_{obs_date}_clean.pkl")
    with open(obs_path, 'rb') as f:
        obs_data = pickle.load(f)[0]
    CD_obs    = obs_data['CD']
    CW_obs    = obs_data['CW']
    lon_obs   = obs_data['lon']
    lat_obs   = obs_data['lat']
    incidents = obs_data['incidents']

    fig = plt.figure(figsize=(13, 8))

    # Forecast axes: 3 columns × 2 rows (CD top, CW bottom)
    axes_fc_cd = [
        fig.add_axes([0.05 + i * 0.18, 0.55, 0.16, 0.35],
                     projection=ccrs.OSGB())
        for i in range(3)
    ]
    axes_fc_cw = [
        fig.add_axes([0.05 + i * 0.18, 0.10, 0.16, 0.35],
                     projection=ccrs.OSGB())
        for i in range(3)
    ]

    # Observation axes
    ax_cd_obs = fig.add_axes([0.65, 0.55, 0.16, 0.35], projection=ccrs.OSGB())
    ax_cw_obs = fig.add_axes([0.65, 0.10, 0.16, 0.35], projection=ccrs.OSGB())

    im_cd = None  # reference for shared forecast colorbar

    # Row labels
    fig.text(0.01, 0.725, "Cold-Dry (CD)",  rotation=90, va='center', fontsize=13)
    fig.text(0.01, 0.275, "Cold-Wet (CW)", rotation=90, va='center', fontsize=13)

    # ── Forecast panels ───────────────────────────────────────────────────────
    for col, entry in enumerate(run_entries):
        lead_time     = entry['lead_time']
        run_date_time = entry['run_date_time']
        tidx          = TIMESTEP_INDEX[lead_time]

        CD = iris.load_cube(
            os.path.join(FORECAST_DIR, f"masked_HadUK_CD_final_{run_date_time}.nc"))
        CW = iris.load_cube(
            os.path.join(FORECAST_DIR, f"masked_HadUK_CW_final_{run_date_time}.nc"))

        # Ensemble mean → probability (%)
        CD_prob = np.mean(CD.data, axis=0) * 100
        CW_prob = np.mean(CW.data, axis=0) * 100

        CD_slice = CD_prob[tidx].copy(); CD_slice[CD_slice == 0] = np.nan
        CW_slice = CW_prob[tidx].copy(); CW_slice[CW_slice == 0] = np.nan

        lons = CD.coord('longitude').points
        lats = CD.coord('latitude').points
        lon_grid, lat_grid = np.meshgrid(lons, lats)

        # CD panel
        ax_cd = axes_fc_cd[col]
        im_cd = ax_cd.pcolormesh(lon_grid, lat_grid, CD_slice,
                                  cmap=cmap_prob, norm=norm_prob, shading='auto',
                                  transform=ccrs.PlateCarree())
        plot_rail(ax_cd)
        set_map_extent(ax_cd)
        ax_cd.set_title(lead_time, fontsize=10)

        # CW panel
        ax_cw = axes_fc_cw[col]
        ax_cw.pcolormesh(lon_grid, lat_grid, CW_slice,
                          cmap=cmap_prob, norm=norm_prob, shading='auto',
                          transform=ccrs.PlateCarree())
        plot_rail(ax_cw)
        set_map_extent(ax_cw)
        ax_cw.set_title(lead_time, fontsize=10)

    # ── Observation panels ────────────────────────────────────────────────────
    for ax_obs, obs_array, cmap_obs, label in [
        (ax_cd_obs, CD_obs, cmap_cd_obs, "Observations"),
        (ax_cw_obs, CW_obs, cmap_cw_obs, "Observations"),
    ]:
        ax_obs.pcolormesh(lon_obs, lat_obs, obs_array,
                          cmap=cmap_obs, norm=norm_obs, shading='auto',
                          transform=ccrs.PlateCarree())
        plot_rail(ax_obs)
        ax_obs.scatter(incidents['Long'], incidents['Lat'],
                       s=incidents['count'] * 10, color='black', alpha=0.6,
                       transform=ccrs.PlateCarree())
        set_map_extent(ax_obs)
        ax_obs.add_feature(cfeature.OCEAN)
        ax_obs.set_title(label, fontsize=10)

    # ── Colorbars ─────────────────────────────────────────────────────────────
    cbar_fc = fig.add_axes([0.60, 0.20, 0.015, 0.50])
    fig.colorbar(im_cd, cax=cbar_fc, label="Forecast Probability (%)")

    cbar_cd = fig.add_axes([0.82, 0.55, 0.015, 0.35])
    cbar1 = fig.colorbar(
        mpl.cm.ScalarMappable(norm=norm_obs, cmap=cmap_cd_obs), cax=cbar_cd)
    cbar1.ax.set_yticks([0, 3])
    cbar1.ax.set_yticklabels(['Neither', 'Cold-Dry'])

    cbar_cw = fig.add_axes([0.82, 0.10, 0.015, 0.35])
    cbar2 = fig.colorbar(
        mpl.cm.ScalarMappable(norm=norm_obs, cmap=cmap_cw_obs), cax=cbar_cw)
    cbar2.ax.set_yticks([0, 3])
    cbar2.ax.set_yticklabels(['Neither', 'Cold-Wet'])

    plt.suptitle(f"Event: {event_name}", fontsize=14)
    out_path = os.path.join(FIG_DIR, f"forecast_prob_{event_name}.png")
    fig.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved {out_path}")


## 3. Produce figures for all events

In [ ]:
for event_name, event_info in EVENTS_DICT.items():
    print(f"\n=== {event_name} ===")
    plot_forecast_event(
        event_name=event_name,
        obs_date=event_info['obs_date'],
        run_entries=event_info['runs'],
    )
